In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

import os

C:\Users\Dipak\AppData\Local\Temp\ipykernel_4656\1312668150.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Dipak\OneDrive\Desktop\MachineLearning\12 ML Projects\Enterprise-RAG-Chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
documents = []

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join("data", file))
        documents.extend(loader.load())

print("Total Pages:", len(documents))

Total Pages: 91


In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 286


In [4]:
print(chunks[0].page_content[:500])

Computer Networks
● Network: A network is a set of devices that are connected with a physical media link. Ina network, two or more nodes are connected by a physical link or two or more networksare connected by one or more nodes. A network is a collection of devices connected toeach other to allow the sharing of data.
● Network Topology:Network topology specifies thelayout of a computer network. Itshows how devices and cables are connected to eachother.
Types of Network Topology:● Star:● Star top


In [5]:
print(chunks[0].metadata)

{'producer': 'Skia/PDF m93 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Computer Networks.docx', 'source': 'data\\Computer Networking Notes for Tech Placements.pdf', 'total_pages': 16, 'page': 0, 'page_label': '1'}


In [6]:
embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding Model Loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5220.52it/s]


Embedding Model Loaded


In [7]:
vector = embedding.embed_query(chunks[0].page_content)

print("Embedding Dimension:", len(vector))

print(vector[:10])

Embedding Dimension: 384
[0.01069464161992073, -0.06895605474710464, -0.018893571570515633, 0.02395515888929367, 0.025050891563296318, 0.0209563747048378, -0.02670426107943058, 0.06018491089344025, 0.04485205560922623, -0.009352536872029305]


In [8]:
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding,
    persist_directory="vectorstore"
)

print("Vector Database Created")
print("Total Documents:", vectordb._collection.count())

Vector Database Created
Total Documents: 858


In [9]:
retriever = vectordb.as_retriever()

results = retriever.invoke("What is self attention?")

print(results[0].page_content)

described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
of a single sequence in order to compute a representation of the sequence. Self-attention has been
used successfully in a variety of tasks including reading comprehension, abstractive summarization,
textual entailment and learning task-independent sentence representations [4, 22, 23, 19].
End-to-end memory networks are based on a recurrent attention mechanism instead of sequence-
aligned recurrence and have been shown to perform well on simple-language question answering and
language modeling tasks [28].
To the best of our knowledge, however, the Transformer is the ﬁrst transduction model relying
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate


In [10]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="gemma3:4b",
    temperature=0
)

print("LLM Loaded")

LLM Loaded


In [11]:
question = "Who is the PM of India?"

docs = retriever.invoke(question)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the context below.

If the answer is not present in the context, say:
"I couldn't find this information in the uploaded documents."

Context:
{context}

Question:
{question}
"""

response = llm.invoke(prompt)

print(response.content)

I couldn't find this information in the uploaded documents.


In [12]:
def ask_question(question):

    docs = retriever.invoke(question)

    context = "\n\n".join([doc.page_content for doc in docs])

    prompt = f"""
You are an Enterprise AI Assistant.

Answer ONLY from the provided context.

Rules:
1. Never use your own knowledge.
2. If the answer is not present in the context, reply:
"I couldn't find this information in the uploaded documents."
3. Keep the answer concise and accurate.

Context:
{context}

Question:
{question}

Answer:
"""

    response = llm.invoke(prompt)

    return response.content

In [13]:
print(ask_question("What is self attention?"))

Self-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. It has been used successfully in tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations.


In [14]:
print(ask_question("What is ACID property?"))

KeyboardInterrupt: 

In [ ]:
print(ask_question("What is TCP?"))

TCPis a connection-oriented protocol, whereasUDPis a connectionlessprotocol.


In [ ]:
print(ask_question("Who is the PM of India?"))

I couldn't find this information in the uploaded documents.


In [ ]:
while True:

    question = input("\nAsk Question: ")

    if question.lower() == "exit":
        break

    print("\nAnswer:")

    print(ask_question(question))


Answer:
The RAG research paradigm represents the earliest methodology, which gained prominence shortly after the emergence of LLMs.
